# History core

> Parse, index and hydrate turn logs without importing the agent runtime.

In [ ]:
#| default_exp history_core

In [ ]:
#| export
import json, math, re
from operator import itemgetter
from pathlib import Path

HISTORY_TAIL = 8_000_000   #: bytes read from the end of a history log
HISTORY_TURNS = 2000       #: turns kept from the history log tail
LEGACY_GAP = 1800          #: seconds between inferred legacy sessions
MAX_SPANS = 256            #: byte ranges an index row lists


def _log_turn(line):
    "One log line as a turn, or `None` when it is blank, not JSON, or not a JSON object."
    if not line.strip(): return None
    try: turn = json.loads(line.decode('utf-8', 'replace'))
    except (AttributeError, RecursionError, ValueError): return None
    return turn if isinstance(turn, dict) else None


def _confined(cfg, ref):
    "A resolved relative path below `cfg`, or `None` when `ref` is not a relative path or a link leads it out of `cfg`."
    if not isinstance(ref, str) or Path(ref).is_absolute(): return None
    root = Path(cfg).resolve()
    try: path = (root/ref).resolve()
    except (OSError, RuntimeError, ValueError): return None   # `RuntimeError` is a symlink loop before Python 3.13
    return path if path.is_relative_to(root) else None


def _details_ref(ref):
    "Whether `ref` names a details file the way a log row does: a folder and a file name, with no `..`."
    return isinstance(ref, str) and len(parts := Path(ref).parts) == 2 and '..' not in parts


def hydrate_turns(cfg, turns):
    "Restore whole results and prompts from confined details files. A missing or unreadable file leaves clipped text."
    files = {}
    for t in turns:
        ref = t.get('details')
        path = _confined(cfg, ref) if cfg is not None and _details_ref(ref) else None
        if path is None: continue
        if ref not in files:
            got, prompts = files[ref] = {}, {}
            try: raw = path.read_bytes()
            except OSError: continue
            for d in map(_log_turn, raw.split(b'\n')):
                if not isinstance(d, dict): continue
                if isinstance(d.get('details'), dict): got.update(d['details'])
                if isinstance(d.get('prompt'), str) and isinstance(d.get('turn_id', ''), str): prompts[d.get('turn_id', '')] = d['prompt']
        got, prompts = files[ref]
        if got and isinstance(t.get('activity'), list):
            t['activity'] = [a | {'detail': got[a['id']]} if isinstance(a, dict) and isinstance(a.get('id'), str) and a['id'] in got else a for a in t['activity']]
        if isinstance(tid := t.get('turn_id', ''), str) and tid in prompts: t['prompt'] = prompts[tid]
    return turns

In [ ]:
#| export
_USER_REQUEST = re.compile(r'<user-request>\n?(.*?)\n?</user-request>', re.S)
_HARNESS = re.compile(r'<(system-reminder|notebook|background-results|watch-results|sent_mid_turn)\b[^>]*>.*?</\1>', re.S)


def request_text(prompt):
    "The person's request, without the notebook or screen context composed around it."
    text = prompt[-1] if isinstance(prompt, (list, tuple)) and prompt else prompt
    text = str(text or '')
    matches = list(_USER_REQUEST.finditer(text))
    return matches[-1].group(1) if matches else text


def request_body(prompt):
    "`request_text` without the harness's blocks, or a `<user-request>` tag a prompt cut short kept."
    return _HARNESS.sub(' ', re.sub(r'</?user-request>', ' ', request_text(prompt))).strip()


def _cut_title(text, n):
    "`text` in at most `n` characters, cut at a word with an ellipsis."
    return text if len(text) <= n else text[:n - 1].rsplit(' ', 1)[0].rstrip(',;:') + '…'


def request_title(prompt, n=80):
    "A session's title from its first request: the first sentence of `request_body`, in at most `n` characters."
    text = ' '.join(request_body(prompt).split()).lstrip('▌│┆┃ ')
    if (match := re.match(r'(.+?[.?!])(\s|$)', text)): text = match.group(1)
    return _cut_title(text, n)

In [ ]:
#| export
def _add_span(spans, start, end):
    "`spans` with `[start, end]` added. The range joins the last one when they touch. Returns None past `MAX_SPANS` or without `spans`."
    if spans is None: return None
    out = [list(span) for span in spans]
    if out and out[-1][1] == start: out[-1][1] = end
    else: out.append([start, end])
    return out if len(out) <= MAX_SPANS else None


def _join_spans(a, b):
    "The ranges of `a` then `b`, or None when either is missing or the sum is too many."
    if a is None or b is None: return None
    for start, end in b:
        if (a := _add_span(a, start, end)) is None: return None
    return a


def _set_spans(row, spans):
    "Store `spans` on `row`, or drop its `spans` when `spans` is None."
    if spans is None: row.pop('spans', None)
    else: row['spans'] = spans


def history_rows(path, offset=0):
    "Yield `(start, end, row)` for each complete object row from `offset`."
    with Path(path).open('rb') as file:
        file.seek(offset)
        for raw in file:
            start, offset = offset, offset + len(raw)
            if not raw.endswith(b'\n'): break
            row = _log_turn(raw)
            if isinstance(row, dict): yield start, offset, row


def _at(row):
    "The finite timestamp of `row`, or `None` when it has none."
    try: at = float(row.get('at') or 0)
    except (OverflowError, TypeError, ValueError): return None
    return at if math.isfinite(at) else None


def labeled_rows(rows, legacy_n=0, legacy_last=None, key=lambda item: item):
    "Yield `(session_id, item)` for each item of `rows` whose row is `key(item)`, with legacy ids inferred and rows with invalid timestamps skipped."
    for item in rows:
        row = key(item)
        if (at := _at(row)) is None: continue
        sid = row.get('session') or ''
        if not sid:
            if legacy_last is None or at - legacy_last > LEGACY_GAP: legacy_n += 1
            sid, legacy_last = f'legacy-{legacy_n}', at
        yield str(sid), item


def _index_rows(rows, legacy_n=0, legacy_last=None):
    "Describe each conversation in `(start, end, row)` values, with byte ranges."
    found = {}
    for sid, (start, end, turn) in labeled_rows(rows, legacy_n, legacy_last, key=itemgetter(2)):
        at = _at(turn)
        if sid not in found:
            found[sid] = {'turns': 0, 'first_at': at, 'first_offset': start,
                          'first_title': request_title(turn.get('prompt', '')), 'spans': []}
        row = found[sid]
        row['turns'] += 1
        row['last_at'], row['last_offset'] = at, end
        row['model'] = turn.get('model', '')
        _set_spans(row, _add_span(row.get('spans'), start, end))
    return found


def _index_from_log(path, offset=0, legacy_n=0, legacy_last=None):
    "Stream a log from `offset` and describe every conversation with its byte ranges."
    return _index_rows(history_rows(path, offset), legacy_n, legacy_last)


def _index_from_history(turns):
    "Describe conversations in in-memory turns without byte ranges."
    found, legacy_n, legacy_last = {}, 0, None
    for sid, turn in labeled_rows(turns):
        at = _at(turn)
        if sid not in found: found[sid] = {'turns': 0, 'first_at': at, 'first_title': request_title(turn.get('prompt', ''))}
        row = found[sid]
        row['turns'] += 1
        row['last_at'], row['model'] = at, turn.get('model', '')
    return found

A history index skips rows whose timestamps are not numeric.

In [ ]:
rows = [
    (0, 10, {'session': 'bad', 'at': 'later', 'prompt': 'skip me'}),
    (10, 20, {'session': 'kept', 'at': 2, 'prompt': 'Keep this turn'}),
]
indexed = _index_rows(rows)
assert list(indexed) == ['kept']
indexed['kept']